# E02:
split up the dataset randomly into 80% train set, 10% dev set, 10% test set. Train the bigram and trigram models only on the training set. Evaluate them on dev and test splits. What can you see?

In [53]:
# import
import random
import torch
import torch.nn.functional as F

In [54]:
# split data set
words = open('../name.txt', 'r').read().splitlines()
random.seed(42)  # make the split reproducible so the two models compare on the same split
random.shuffle(words)
n_words = len(words)
n_train = int(0.8 * n_words)
n_dev = int(0.1 * n_words)
train_words = words[:n_train]
dev_words = words[n_train: (n_train + n_dev)]
test_words = words[(n_train + n_dev):]
print(len(train_words), len(dev_words), len(test_words))

# build stoi and itos
chars = sorted(list(set("".join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi["."] = 0
itos = {i: s for s, i in stoi.items()}

25626 3203 3204


In [66]:
# bigram model
def parse_bigram_data(words):
    # create the training set of bigrams
    xs, ys = [], []
    for w in words:
        chs = ["."] + list(w) + ["."]
        for ch1, ch2 in zip(chs, chs[1:]):
            ix1 = stoi[ch1]
            ix2 = stoi[ch2]
            xs.append(ix1)
            ys.append(ix2)

    xs = torch.tensor(xs)
    ys = torch.tensor(ys)

    return xs, ys

def train_bigram_model(n_iter=100):

    xs, ys = parse_bigram_data(train_words)
    n_train_data = len(xs)

    # initialize the network
    g = torch.Generator().manual_seed(2147483647)
    W = torch.randn((27, 27), generator=g, requires_grad=True)

    # gradient descent
    for k in range(n_iter):
        # forward pass
        xenc = F.one_hot(xs, num_classes=27).float()
        logits = xenc @ W  # interpreted as log-counts; predict the log-counts
        counts = logits.exp()  # equivalent to N
        probs = counts / counts.sum(1, keepdim=True)  # probabilities for next character
        # L2 regularization
        # loss = -probs[torch.arange(n_train_data), ys].log().mean() + 0.1 * (W**2).mean()
        loss = -probs[torch.arange(n_train_data), ys].log().mean()
        if k == n_iter - 1:
            print("train loss: ", loss.item())

        # backward pass
        W.grad = None  # set to zero the gardient
        loss.backward()

        # update
        W.data += -50 * W.grad

    return W


@torch.no_grad()
def evaluate_bigram_model(W, evaluate_words):
    xs, ys = parse_bigram_data(evaluate_words)
    n = len(xs)

    xenc = F.one_hot(xs, num_classes=27).float()
    logits = xenc @ W  # interpreted as log-counts; predict the log-counts
    counts = logits.exp()  # equivalent to N
    probs = counts / counts.sum(1, keepdim=True)  # probabilities for next character
    loss = -probs[torch.arange(n), ys].log().mean()

    return loss.item()


In [69]:
# trigram model

def parse_trigram_data(words):
    xs, ys, zs = [], [], []
    for w in words:
        chs = ["."] + ["."] + list(w) + ["."]
        for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
            ix1 = stoi[ch1]
            ix2 = stoi[ch2]
            ix3 = stoi[ch3]
            xs.append(ix1)
            ys.append(ix2)
            zs.append(ix3)

    xs = torch.tensor(xs)
    ys = torch.tensor(ys)
    zs = torch.tensor(zs)
    return xs, ys, zs

# create the training set of trigrams
def train_trigram_model(n_iter=500):
    xs, ys, zs = parse_trigram_data(train_words)
    n_train_data = len(xs)

    # initialize the network
    g = torch.Generator().manual_seed(2147483647)
    W = torch.randn((27, 27, 27), generator=g, requires_grad=True)

    # gradient descent
    # the trigram has 27x more parameters (27^3) than the bigram, and each of the
    # 729 context slices only sees a sparse share of the gradient, so it needs far
    # more iterations than the bigram to actually fit the training set.
    for k in range(n_iter):
        # forward pass
        # W3[xs, ys] is advanced indexing: for every example i it plucks the row
        # W3[xs[i], ys[i], :], producing (num, 27) logits. this is the learned
        # analogue of looking up N3[ix1, ix2] in the first approach.
        logits = W[xs, ys]  # (num, 27) log-counts
        counts = logits.exp()  # (num, 27) counts ~ N3
        probs = counts / counts.sum(1, keepdim=True)  # (num, 27) softmax -> P3
        # L2 regularization
        # loss = -probs[torch.arange(n_train_data), zs].log().mean() + 0.1 * (W**2).mean()
        loss = -probs[torch.arange(n_train_data), zs].log().mean()
        if k == n_iter - 1:
            print("train loss: ", loss.item())

        # backward pass
        W.grad = None
        loss.backward()

        # update
        W.data += -50 * W.grad
    
    return W

@torch.no_grad()
def evaluate_trigram_model(W, evaluate_words):
    xs, ys, zs = parse_trigram_data(evaluate_words)
    n = len(xs)

    logits = W[xs, ys]  # (num, 27) log-counts
    counts = logits.exp()  # (num, 27) counts ~ N3
    probs = counts / counts.sum(1, keepdim=True)  # (num, 27) softmax -> P3
    loss = -probs[torch.arange(n), zs].log().mean()

    return loss.item()



In [71]:
# evalute and compare
print("bigram model results--------")
W_bi = train_bigram_model(n_iter=100)
print("loss on dev words: ", evaluate_bigram_model(W_bi, dev_words))
print("loss on test words: ", evaluate_bigram_model(W_bi, test_words))
print("\n")

print("trigram model results--------")
W_tri = train_trigram_model(n_iter=400)
print("loss on dev words: ", evaluate_trigram_model(W_tri, dev_words))
print("loss on test words: ", evaluate_trigram_model(W_tri, test_words))

bigram model results--------
train loss:  2.472895860671997
loss on dev words:  2.4705991744995117
loss on test words:  2.4771249294281006


trigram model results--------
train loss:  2.296832799911499
loss on dev words:  2.312957525253296
loss on test words:  2.3107218742370605
